In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
accounts = spark.table("aml_framework.accounts")
alerts = spark.table("aml_framework.alerts")
transactions = spark.table("aml_framework.transactions")
saml_d = spark.table("aml_framework.saml_d")

# IBM AMLSim

## Accounts

In [0]:
accounts.columns

### Primary Key

In [0]:
acct_pk = ["ACCOUNT_ID"]
if accounts.count() == accounts.select(acct_pk).drop_duplicates().count():
    print("Primary key is ", acct_pk)
else:
    print("Primary key not found")

### Duplicate rows

In [0]:
if accounts.count() == accounts.drop_duplicates().count():
    print("All rows in accounts are unique")
else:
    print("There are duplicate rows in accounts")

### Nulls

In [0]:
for col in accounts.columns:
    print("Count of nulls in column", col, "is" , accounts.filter(accounts[col].isNull()).count())

### Distribution

In [0]:
def column_profile(df, col_name):
    print(f"\nColumn: {col_name}")

    df.select(
        F.count(col_name).alias("count"),
        F.mean(col_name).alias("mean"),
        F.stddev(col_name).alias("stddev"),
        F.min(col_name).alias("min"),
        F.max(col_name).alias("max"),
        F.skewness(col_name).alias("skewness")
    ).show()

    q = df.approxQuantile(
        col_name,
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99],
        0.01
    )

    print(
        "P1, P5, P25, P50, P75, P95, P99:\n",
        q
    )

for col in accounts.columns:
    if ((accounts.schema[col].dataType != StringType()) & (accounts.schema[col].dataType != BooleanType()) & (col not in acct_pk) & ('ID' not in col)):
        # print(accounts.schema[col].dataType)
        column_profile(accounts, col)

### Outliers

In [0]:
def column_profile_with_outliers(df, col_name, pk_col="acct_pk"):
    print(f"\n{'='*80}")
    print(f"Column: {col_name}")

    # Summary stats
    df.select(
        F.count(col_name).alias("count"),
        F.mean(col_name).alias("mean"),
        F.stddev(col_name).alias("stddev"),
        F.min(col_name).alias("min"),
        F.max(col_name).alias("max"),
        F.skewness(col_name).alias("skewness")
    ).show(truncate=False)

    # Percentiles
    q = df.approxQuantile(
        col_name,
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99],
        0.01
    )

    p1, p5, q1, p50, q3, p95, p99 = q

    print(f"P1={p1}, P5={p5}, P25={q1}, P50={p50}, P75={q3}, P95={p95}, P99={p99}")

    # IQR Outlier Detection
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers_df = df.filter(
        (F.col(col_name) < lower_bound) |
        (F.col(col_name) > upper_bound)
    )

    outlier_count = outliers_df.count()

    print(f"IQR Lower Bound: {lower_bound}")
    print(f"IQR Upper Bound: {upper_bound}")
    print(f"Outlier Count: {outlier_count}")

    if outlier_count > 0:
        print(f"Sample outlier {pk_col}s:")

        outliers_df.select(
            *pk_col,
            col_name
        ).orderBy(
            F.abs(F.col(col_name)).desc()
        ).show(20, truncate=False)

    return outliers_df

In [0]:
outlier_results = {}

for col in accounts.columns:

    if (
        not isinstance(accounts.schema[col].dataType, (StringType, BooleanType))
        and col not in acct_pk
        and "ID" not in col.upper()
    ):

        outlier_df = column_profile_with_outliers(
            accounts,
            col,
            pk_col=acct_pk
        )

        if outlier_df.count() > 0:
            outlier_results[col] = outlier_df

## Alerts

In [0]:
alerts.columns

### Primary Keys

In [0]:
alert_pk = ["ALERT_ID", "TX_ID"]
if alerts.count() == alerts.select(alert_pk).drop_duplicates().count():
    print("Primary key is ", alert_pk)
else:
    print("Primary key not found")

### Duplicate rows

In [0]:
if alerts.count() == alerts.drop_duplicates().count():
    print("All rows in alerts are unique")
else:
    print("There are duplicate rows in alerts")

### Nulls

In [0]:
for col in alerts.columns:
    print("Count of nulls in column", col, "is" , alerts.filter(alerts[col].isNull()).count())

### Distribution

In [0]:
for col in alerts.columns:
    if ((alerts.schema[col].dataType != StringType()) & (alerts.schema[col].dataType != BooleanType()) & (col not in alert_pk) & ('ID' not in col)):
        if col != 'TIMESTAMP':
            column_profile(alerts, col)

### Outliers

In [0]:
outlier_results = {}

for col in alerts.columns:

    if (
        not isinstance(alerts.schema[col].dataType, (StringType, BooleanType))
        and col not in acct_pk
        and "ID" not in col.upper()
        and col != "TIMESTAMP"
    ):

        outlier_df = column_profile_with_outliers(
            alerts,
            col,
            pk_col=alert_pk
        )

        if outlier_df.count() > 0:
            outlier_results[col] = outlier_df

## Transactions

In [0]:
transactions.columns

### Primary Key

In [0]:
trxn_pk = ["TX_ID"]
if transactions.count() == transactions.select(trxn_pk).drop_duplicates().count():
    print("Primary key is ", trxn_pk)
else:
    print("Primary key not found")

### Duplicate rows

In [0]:
if transactions.count() == transactions.drop_duplicates().count():
    print("All rows in transactions are unique")
else:
    print("There are duplicate rows in transactions")

### Nulls

In [0]:
for col in transactions.columns:
    print("Count of nulls in column", col, "is" , transactions.filter(transactions[col].isNull()).count())

### Distribution

In [0]:
for col in transactions.columns:
    if ((transactions.schema[col].dataType != StringType()) & (transactions.schema[col].dataType != BooleanType()) & (col not in trxn_pk) & ('ID' not in col)):
        if col != 'TIMESTAMP':
            column_profile(transactions, col)

### Outliers

In [0]:
outlier_results = {}

for col in transactions.columns:

    if (
        not isinstance(transactions.schema[col].dataType, (StringType, BooleanType))
        and col not in trxn_pk
        and "ID" not in col.upper()
        and col != "TIMESTAMP"
    ):

        trxn_outlier_df = column_profile_with_outliers(
            transactions,
            col,
            pk_col=trxn_pk
        )

        if trxn_outlier_df.count() > 0:
            outlier_results[col] = trxn_outlier_df

In [0]:
trxn_outlier_df.filter("IS_FRAUD == true").display()

### Joins

In [0]:
alerts.withColumnRenamed("TX_AMOUNT", "ALERT_TX_AMOUNT").join(transactions, on=["TX_ID", "ALERT_ID"], how="left").filter("TX_AMOUNT == 1047234").display()

# SAML-D

In [0]:
saml_d.columns

### Primary Key

In [0]:
samld_pk = ["Sender_account", "Time", "Amount"]
if saml_d.drop_duplicates().count() == saml_d.select(samld_pk).drop_duplicates().count():
    print("Primary key is ", samld_pk)
else:
    print("Primary key not found")

### Duplicate rows

In [0]:
if saml_d.count() == saml_d.drop_duplicates().count():
    print("All rows in saml_d are unique")
else:
    print("There are duplicate rows in saml_d")

### Nulls

In [0]:
for col in saml_d.columns:
    print("Count of nulls in column", col, "is" , saml_d.filter(saml_d[col].isNull()).count())

### Distribution

In [0]:
for col in saml_d.columns:
    if ((saml_d.schema[col].dataType != StringType()) & (saml_d.schema[col].dataType != BooleanType()) & (col not in list(set(samld_pk)-set(["Amount"]))) & ('ID' not in col)):
        if (col not in ['Date', 'Is_laundering', 'Receiver_account']):
            column_profile(saml_d, col)

### Outliers

In [0]:
outlier_results = {}

for col in saml_d.columns:

    if (
        not isinstance(saml_d.schema[col].dataType, (StringType, BooleanType))
        and col not in list(set(samld_pk)-set(["Amount"]))
        and "ID" not in col.upper()
        and col not in ['Date', 'Is_laundering', 'Receiver_account']
    ):

        saml_d_outlier_df = column_profile_with_outliers(
            saml_d,
            col,
            pk_col=samld_pk
        )

        if saml_d_outlier_df.count() > 0:
            outlier_results[col] = saml_d_outlier_df

In [0]:
saml_d_outlier_df.filter("Is_laundering == 1").display()